In [1]:
import os
import sys
import pandas as pd
from typing import Tuple

In [2]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [3]:
from utility.data_log_functions import DataLogHelper

In [4]:
def compare_multiple_code_generation_logs(res_dir: str, filter: Tuple[str] = (), anti_filter: Tuple[str] = ()):
    
    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (
            os.path.isfile(os.path.join(res_dir, f)) and 
            f.endswith(".csv") and 
            all(sub in f for sub in filter)) and
            all(sub not in f for sub in anti_filter)
            ]

    log_file_names = [csv_file_name.replace('.csv', '') for csv_file_name in csv_logs]

    results_df = pd.DataFrame(columns=log_file_names, index = log_file_names)
    for file_name in log_file_names:
        results_df.loc[file_name, file_name] = float('nan')

    while len(csv_logs) > 0:
        log1_file_name = csv_logs.pop()
        for log2_file_name in csv_logs:
            log1_file_path = os.path.join(res_dir, log1_file_name)
            log2_file_path = os.path.join(res_dir, log2_file_name)

            log1 = pd.read_csv(log1_file_path)
            log2 = pd.read_csv(log2_file_path) 
            print(log1_file_name, log2_file_name)
            log1_inconsistencies, log2_inconsistencies = DataLogHelper.compare_code_generation_dataframe_results(log1=log1, log2=log2)

            results_df.loc[log1_file_name.replace('.csv', ''), log2_file_name.replace('.csv', '')] = log1_inconsistencies
            results_df.loc[log2_file_name.replace('.csv', ''), log1_file_name.replace('.csv', '')] = log2_inconsistencies

    return results_df

In [ ]:
def compare_logs_against_no_mutation(res_dir: str, filter: Tuple[str] = (), anti_filter: Tuple[str] = ()):
    
    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (
            os.path.isfile(os.path.join(res_dir, f)) and 
            f.endswith(".csv") and 
            all(sub in f for sub in filter)) and
            all(sub not in f for sub in anti_filter)
            ]

    print(csv_logs)
    target_log_name = [l for l in csv_logs if "no_mutation" in l][-1]
    print(target_log_name)
    csv_logs.pop(csv_logs.index(target_log_name))
    target_log_path = os.path.join(res_dir, target_log_name)
    target_log = pd.read_csv(target_log_path)

    results_df = pd.DataFrame()

    total_inconsistencies = 0
    total_questions = 0

    for log_name in csv_logs:
        print(target_log_name, log_name)
        log2_file_path = os.path.join(res_dir, log_name)
        log2 = pd.read_csv(log2_file_path) 

        inconsistency_dict = DataLogHelper.compare_code_generation_dataframe_results(log1=target_log, log2=log2)

        results_df.loc[log_name.replace('.csv', ''), "Inconsistency Score"] = f"{inconsistency_dict['log1_inconsistencies'] + inconsistency_dict['log2_inconsistencies']}/{inconsistency_dict['total_questions']} = {round((inconsistency_dict['log1_inconsistencies'] + inconsistency_dict['log2_inconsistencies'])*100/inconsistency_dict['total_questions'], 2)}"
        results_df.loc[log_name.replace('.csv', ''), "Inconsistency Score (%)"] = f"{round((inconsistency_dict['log1_inconsistencies'] + inconsistency_dict['log2_inconsistencies'])*100/inconsistency_dict['total_questions'], 2)}"

        total_inconsistencies += inconsistency_dict['log1_inconsistencies'] + inconsistency_dict['log2_inconsistencies']
        total_questions += inconsistency_dict['total_questions']

    results_df.loc["Aggregated Results", "Inconsistency Score"] = f"{total_inconsistencies}/{}{round(total_inconsistencies*100/total_questions, 2)}"
    results_df.loc["Aggregated Results", "Inconsistency Score (%)"] = round(total_inconsistencies*100/total_questions, 2)

    return results_df

In [6]:
res_dir = proj_dir + "/results/code_generation/mistral"
res_dir = "/Users/jin/Downloads/Test_results_26:09:2025/input_prediction/gpt-4o"
# res_dir = "/Users/jin/Downloads/gemma-3-12b-it_(output_prediction)"
res = compare_logs_against_no_mutation(res_dir=res_dir, filter=("CruxEval", ))
res_df = pd.DataFrame(res)

print(res_df)

['CruxEval_zero_shot_literal_format.csv', 'CruxEval_zero_shot_constant_unfold_add.csv', 'CruxEval_zero_shot_random.csv', 'CruxEval_zero_shot_for2enumerate.csv', 'CruxEval_zero_shot_boolean_literal.csv', 'CruxEval_zero_shot_no_mutation.csv', 'CruxEval_zero_shot_constant_unfold.csv', 'CruxEval_zero_shot_commutative_reorder_new.csv', 'CruxEval_zero_shot_constant_unfold_mult.csv', 'CruxEval_zero_shot_for2while.csv', 'CruxEval_zero_shot_demorgan.csv', 'CruxEval_zero_shot_sequential.csv']
CruxEval_zero_shot_no_mutation.csv
CruxEval_zero_shot_no_mutation.csv CruxEval_zero_shot_literal_format.csv
Starting comparison of 800 tasks...

=== COMPARISON SUMMARY ===
Total tasks processed: 800
Both succeeded: 255
Both failed: 0
IdenticalMutationError: 0
Log1 Assertion Errors 154
Log2 Assertion Errors 75
Comparable tasks (atleast one succeeded): 278
  - Log1 failed, Log2 succeeded: 12
  - Log1 succeeded, Log2 failed: 11
Total inconsistencies: 23/278
CruxEval_zero_shot_no_mutation.csv CruxEval_zero_shot